# E0 — MLP trên landmark thô


## 2. Import + cấu hình

In [1]:
import re
import json
import time
from pathlib import Path
from typing import Protocol

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang chạy trên: {DEVICE}")

# Dataset: https://www.kaggle.com/datasets/hauuto/vietnamese-sign-language-alphabet
# Tự dò tìm thư mục landmarks/landmarks/raw trong /kaggle/input — không phụ thuộc phải biết
# trước đường dẫn chính xác, vì Kaggle có thể đưa dataset vào /kaggle/input/<slug>/... hoặc
# /kaggle/input/datasets/<username>/<slug>/... tuỳ cách add.
_candidates = list(Path("/kaggle/input").rglob("landmarks/landmarks/raw"))
if not _candidates:
    raise FileNotFoundError(
        "Không tìm thấy landmarks/landmarks/raw trong /kaggle/input — "
        "kiểm tra lại đã Add Data đúng dataset vietnamese-sign-language-alphabet chưa."
    )
LANDMARK_DIR = _candidates[0]
print(f"Dùng LANDMARK_DIR = {LANDMARK_DIR}")

PEOPLE = ("hau", "khoi", "tai", "vy")
FNAME_RE = re.compile(r"^([a-z_]+)_([a-z]+)_([AB])_(\d+)\.npy$")

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("/kaggle/working/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_PATH = Path("/kaggle/working/E0_results.json")

FRAME_INDEX = 22  # khung cố định thứ 23/45, đã chốt trong kế hoạch nhóm cho E0/E1
EPOCHS = 100
LR = 1e-3
BATCH_SIZE = 32
VAL_RATIO = 0.15  # tách từ tập train để theo dõi val acc, không đụng vào tập test cross-subject

MAU_XANH = "#2a78d6"
MAU_CAM = "#eb6834"


Đang chạy trên: cuda
Dùng LANDMARK_DIR = /kaggle/input/datasets/hauuto/vietnamese-sign-language-alphabet/landmarks/landmarks/raw


## 3. Đọc dữ liệu landmark

In [2]:
def load_all_landmarks():
    records = []
    for f in sorted(LANDMARK_DIR.glob("*/*.npy")):
        m = FNAME_RE.match(f.name)
        if not m:
            print(f"CẢNH BÁO: tên file không đúng quy ước: {f.name}")
            continue
        code, person, block, seq = m.groups()
        arr = np.load(f)
        records.append({"code": code, "person": person, "block": block, "seq": int(seq), "arr": arr})
    return records


records = load_all_landmarks()
print(f"Tổng số mẫu đọc được: {len(records)}")

ALL_CLASSES = sorted(set(r["code"] for r in records))
CODE_TO_IDX = {c: i for i, c in enumerate(ALL_CLASSES)}
IDX_TO_CODE = {i: c for c, i in CODE_TO_IDX.items()}
print(f"Số lớp: {len(ALL_CLASSES)}")


Tổng số mẫu đọc được: 640
Số lớp: 34


## 4. Khung đánh giá cross-subject (dùng chung cho cả 4 thực nghiệm)

In [3]:
# Khung đánh giá cross-subject — giống hệt src/cross_subject.py, copy nguyên để notebook
# chạy độc lập trên Kaggle (không phụ thuộc phải có repo).
class ModelStrategy(Protocol):
    def prepare_input(self, X: np.ndarray) -> np.ndarray: ...
    def train(self, X_train, y_train, tag: str): ...
    def predict(self, model_state, X_test) -> np.ndarray: ...
    def measure_latency(self, model_state, X_sample) -> float: ...


def run_cross_subject(records, strategy: ModelStrategy, people=PEOPLE):
    results = []
    for test_person in people:
        train_records = [r for r in records if r["person"] != test_person]
        test_records = [r for r in records if r["person"] == test_person]

        X_train_raw = np.stack([r["arr"] for r in train_records])
        y_train = np.array([CODE_TO_IDX[r["code"]] for r in train_records])
        X_test_raw = np.stack([r["arr"] for r in test_records])
        y_test = np.array([CODE_TO_IDX[r["code"]] for r in test_records])

        X_train = strategy.prepare_input(X_train_raw)
        X_test = strategy.prepare_input(X_test_raw)

        model_state = strategy.train(X_train, y_train, tag=test_person)
        y_pred = strategy.predict(model_state, X_test)
        accuracy = float(np.mean(y_pred == y_test))
        latency_ms = strategy.measure_latency(model_state, X_test[:1])

        results.append({"test_person": test_person, "accuracy": accuracy, "latency_ms": latency_ms})
        print(f"Test trên {test_person}: accuracy={accuracy:.3f}, latency={latency_ms:.2f}ms")

    accs = [r["accuracy"] for r in results]
    lats = [r["latency_ms"] for r in results]
    print(f"\nTrung bình: accuracy={np.mean(accs):.3f} (±{np.std(accs):.3f}), "
          f"latency={np.mean(lats):.2f}ms (±{np.std(lats):.2f}ms)")
    return results


## 5. Hàm vẽ biểu đồ train loss / val accuracy

In [4]:
def plot_history(history, tag: str, fold_idx: int, exp: str = "E0"):
    fig, ax1 = plt.subplots(figsize=(7, 4.5))

    ax1.plot(history["epoch"], history["train_loss"], color=MAU_XANH, linewidth=2, label="Train loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Train loss", color=MAU_XANH)
    ax1.tick_params(axis="y", labelcolor=MAU_XANH)

    ax2 = ax1.twinx()
    ax2.plot(history["epoch"], history["val_acc"], color=MAU_CAM, linewidth=2, label="Val accuracy")
    ax2.set_ylabel("Val accuracy", color=MAU_CAM)
    ax2.tick_params(axis="y", labelcolor=MAU_CAM)
    ax2.set_ylim(0, 1)

    fig.suptitle(f"{exp} — Train loss & Val accuracy (fold test={tag})")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")

    fig.tight_layout()
    out_path = PLOT_DIR / f"{exp}_fold{fold_idx}_{tag}.png"
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"  Đã lưu biểu đồ: {out_path}")


## 6. Kiến trúc mô hình MLP

In [5]:
# Kiến trúc MLP — dùng chung cấu trúc mạng cho cả E0 và E1 để so sánh công bằng
class MLPClassifier(nn.Module):
    def __init__(self, in_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)


## 7. Strategy cho E0

In [6]:
# Strategy cho E0 — MLP trên landmark THÔ, khung cố định index=22, KHÔNG chuẩn hóa
class E0Strategy:
    def __init__(self):
        self.fold_idx = 0  # để đặt tên checkpoint/biểu đồ khác nhau cho từng lượt xoay vòng

    def prepare_input(self, X: np.ndarray) -> np.ndarray:
        # (N, 45, 63) -> (N, 63): lấy đúng khung cố định thứ 23/45, không chuẩn hóa toạ độ (đúng định nghĩa E0)
        return X[:, FRAME_INDEX, :]

    def train(self, X_train, y_train, tag: str):
        # Tách val từ train (stratified đơn giản theo lớp) — chỉ để theo dõi, không ảnh hưởng
        # đến việc mô hình được train trên toàn bộ 3 người còn lại khi đánh giá cuối cùng.
        rng = np.random.default_rng(42)
        val_idx, train_idx = [], []
        for c in np.unique(y_train):
            idx = np.where(y_train == c)[0]
            rng.shuffle(idx)
            n_val = max(1, int(len(idx) * VAL_RATIO))
            val_idx.extend(idx[:n_val])
            train_idx.extend(idx[n_val:])
        train_idx, val_idx = np.array(train_idx), np.array(val_idx)

        X_tr, y_tr = X_train[train_idx], y_train[train_idx]
        X_val, y_val = X_train[val_idx], y_train[val_idx]

        in_dim = X_tr.shape[1]
        num_classes = len(ALL_CLASSES)
        model = MLPClassifier(in_dim, num_classes).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        criterion = nn.CrossEntropyLoss()

        X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
        y_tr_t = torch.tensor(y_tr, dtype=torch.long)
        loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=BATCH_SIZE, shuffle=True)

        X_val_t = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
        y_val_t = torch.tensor(y_val, dtype=torch.long).to(DEVICE)

        history = {"epoch": [], "train_loss": [], "val_acc": []}

        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0.0
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()
                out = model(xb)
                loss = criterion(out, yb)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * xb.size(0)
            train_loss = total_loss / len(X_tr)

            model.eval()
            with torch.no_grad():
                val_pred = torch.argmax(model(X_val_t), dim=1)
                val_acc = float((val_pred == y_val_t).float().mean())

            history["epoch"].append(epoch + 1)
            history["train_loss"].append(train_loss)
            history["val_acc"].append(val_acc)

            if (epoch + 1) % 20 == 0:
                print(f"  [E0 fold {self.fold_idx}] epoch {epoch+1}/{EPOCHS} "
                      f"loss={train_loss:.4f} val_acc={val_acc:.3f}")

        ckpt_path = CHECKPOINT_DIR / f"E0_fold{self.fold_idx}.pt"
        torch.save(model.state_dict(), ckpt_path)
        print(f"  Đã lưu checkpoint: {ckpt_path}")

        plot_history(history, tag, self.fold_idx)
        self.fold_idx += 1

        model.eval()
        return model
    def predict(self, model_state, X_test) -> np.ndarray:
        model_state.eval()
        with torch.no_grad():
            X_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
            logits = model_state(X_t)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
        return preds

    def measure_latency(self, model_state, X_sample) -> float:
        # Đo trên CPU để phản ánh đúng điều kiện suy luận thực tế lúc demo (không phải lúc train trên GPU Kaggle)
        model_cpu = model_state.to("cpu").eval()
        x = torch.tensor(X_sample, dtype=torch.float32)

        with torch.no_grad():
            for _ in range(5):  # warmup, không tính vào kết quả
                model_cpu(x)
            times = []
            for _ in range(50):
                t0 = time.perf_counter()
                model_cpu(x)
                times.append((time.perf_counter() - t0) * 1000)

        model_state.to(DEVICE)  # trả model về lại device cũ để không ảnh hưởng lượt sau
        return float(np.mean(times))


## 8. Chạy đánh giá cross-subject

In [7]:
strategy = E0Strategy()
results = run_cross_subject(records, strategy)

with open(RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "experiment": "E0",
        "description": "MLP tren landmark tho, khung co dinh index=22, khong chuan hoa toa do",
        "results": results,
        "accuracy_mean": float(np.mean([r["accuracy"] for r in results])),
        "accuracy_std": float(np.std([r["accuracy"] for r in results])),
        "latency_mean_ms": float(np.mean([r["latency_ms"] for r in results])),
    }, f, ensure_ascii=False, indent=2)

print(f"\nĐã lưu kết quả vào: {RESULT_PATH}")
print("Các file cần tải về cho báo cáo (B7): checkpoints/E0_fold*.pt, plots/E0_fold*.png, "
      "E0_results.json, notebook này (File > Download)")


  [E0 fold 0] epoch 20/100 loss=2.6789 val_acc=0.304
  [E0 fold 0] epoch 40/100 loss=2.0008 val_acc=0.500
  [E0 fold 0] epoch 60/100 loss=1.7329 val_acc=0.522
  [E0 fold 0] epoch 80/100 loss=1.5353 val_acc=0.543
  [E0 fold 0] epoch 100/100 loss=1.4203 val_acc=0.565
  Đã lưu checkpoint: /kaggle/working/checkpoints/E0_fold0.pt
  Đã lưu biểu đồ: /kaggle/working/plots/E0_fold0_hau.png
Test trên hau: accuracy=0.362, latency=0.06ms
  [E0 fold 1] epoch 20/100 loss=2.9474 val_acc=0.261
  [E0 fold 1] epoch 40/100 loss=2.3524 val_acc=0.457
  [E0 fold 1] epoch 60/100 loss=1.9556 val_acc=0.413
  [E0 fold 1] epoch 80/100 loss=1.7852 val_acc=0.435
  [E0 fold 1] epoch 100/100 loss=1.6086 val_acc=0.478
  Đã lưu checkpoint: /kaggle/working/checkpoints/E0_fold1.pt
  Đã lưu biểu đồ: /kaggle/working/plots/E0_fold1_khoi.png
Test trên khoi: accuracy=0.450, latency=0.07ms
  [E0 fold 2] epoch 20/100 loss=2.8346 val_acc=0.152
  [E0 fold 2] epoch 40/100 loss=2.2845 val_acc=0.326
  [E0 fold 2] epoch 60/100 loss=